# PostFab LLM Fine-tuning (QLoRA)

반도체 후공정 P&T 도메인 특화 언어모델 파인튜닝 실험.

## 목표
- `data/finetune/postfab_qa.jsonl` 데이터셋으로 LLM 파인튜닝
- QLoRA (4-bit quantization + LoRA)로 소비 메모리 최소화
- 파인튜닝 전/후 도메인 용어 응답 품질 비교

## 환경 요구사항
- GPU: NVIDIA CUDA 지원 (Google Colab T4 이상 권장)
- RAM: 12GB 이상
- VRAM: 8GB 이상

## 1. 패키지 설치

In [ ]:
# Colab 환경에서 실행 시 주석 해제
# !pip install -q transformers datasets peft bitsandbytes accelerate trl

## 2. 데이터셋 로드 및 확인

In [ ]:
import json
import os

# 경로 설정 (Colab에서는 직접 업로드 또는 경로 수정)
DATA_PATH = "../data/finetune/postfab_qa.jsonl"

samples = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        samples.append(json.loads(line.strip()))

print(f"총 샘플 수: {len(samples)}")
print("\n첫 번째 샘플:")
print(json.dumps(samples[0], ensure_ascii=False, indent=2))

## 3. Hugging Face Dataset 변환

In [ ]:
from datasets import Dataset

# Chat format → 단일 텍스트로 변환 (SFT용)
def format_sample(sample):
    msgs = sample["messages"]
    user_msg = next(m["content"] for m in msgs if m["role"] == "user")
    assistant_msg = next(m["content"] for m in msgs if m["role"] == "assistant")
    return {
        "text": f"### 질문\n{user_msg}\n\n### 답변\n{assistant_msg}"
    }

formatted = [format_sample(s) for s in samples]
dataset = Dataset.from_list(formatted)
print(dataset)
print("\n예시:")
print(dataset[0]["text"])

## 4. 베이스 모델 및 토크나이저 로드

사용 모델: `beomi/Llama-3-Open-Ko-8B` (한국어 특화 Llama-3 8B)  
리소스 부족 시 대안: `Qwen/Qwen2.5-1.5B-Instruct`

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "beomi/Llama-3-Open-Ko-8B"
# 경량 대안: MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit 양자화 설정 (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"모델 로드 완료: {MODEL_ID}")
print(f"디바이스: {model.device}")

## 5. LoRA 어댑터 설정

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                       # LoRA rank
    lora_alpha=32,              # 스케일링 factor
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. 파인튜닝 전 베이스라인 평가

In [ ]:
def generate_response(model, tokenizer, question, max_new_tokens=256):
    prompt = f"### 질문\n{question}\n\n### 답변\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# 파인튜닝 전 테스트
test_questions = [
    "FDC가 뭐야?",
    "tRCD Timing Fail은 어떤 경우에 발생해?",
]

print("=" * 60)
print("[파인튜닝 전 베이스라인]")
print("=" * 60)
for q in test_questions:
    print(f"\n질문: {q}")
    print(f"답변: {generate_response(model, tokenizer, q)}")
    print("-" * 40)

## 7. SFT 학습

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./postfab-qlora-output"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_steps=50,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    dataset_text_field="text",
    max_seq_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    tokenizer=tokenizer,
)

print("학습 시작...")
trainer.train()
print("학습 완료!")

## 8. 파인튜닝 후 평가 및 비교

In [ ]:
print("=" * 60)
print("[파인튜닝 후 결과]")
print("=" * 60)
for q in test_questions:
    print(f"\n질문: {q}")
    print(f"답변: {generate_response(model, tokenizer, q)}")
    print("-" * 40)

## 9. LoRA 어댑터 저장

In [ ]:
ADAPTER_DIR = "./postfab-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA 어댑터 저장 완료: {ADAPTER_DIR}")

# 저장된 파일 확인
import os
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f"  {f}: {size/1024/1024:.1f} MB")

## 10. 저장된 어댑터 로드 및 추론 테스트

학습 완료 후 별도 환경에서 어댑터만 로드하여 사용하는 방법.

In [ ]:
from peft import PeftModel

# 베이스 모델 로드 (양자화 적용)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

# LoRA 어댑터 병합
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
ft_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

print("파인튜닝 모델 로드 완료")

# 추론 테스트
test_q = "수율 저하 원인을 분석하려면 어떤 데이터를 봐야 해?"
print(f"\n질문: {test_q}")
print(f"답변: {generate_response(ft_model, ft_tokenizer, test_q)}")

## 부록: Claude API Fine-tuning (대안)

로컬 GPU가 없는 경우, Anthropic의 파인튜닝 API를 사용할 수 있습니다.  
현재 claude-haiku-3 모델 파인튜닝을 지원합니다.

In [ ]:
# Claude API Fine-tuning 예시 (참고용)
# 실제 사용 시 Anthropic 파인튜닝 API 접근 권한 필요

CLAUDE_FINETUNE_EXAMPLE = """
import anthropic

client = anthropic.Anthropic()

# 1. 파인튜닝 작업 생성
job = client.fine_tuning.jobs.create(
    model="claude-haiku-3",
    training_file="postfab_qa.jsonl",  # 업로드된 파일 ID
    hyperparameters={"n_epochs": 3}
)

# 2. 상태 확인
status = client.fine_tuning.jobs.retrieve(job.id)
print(f"Status: {status.status}")

# 3. 파인튜닝된 모델로 추론
response = client.messages.create(
    model=status.fine_tuned_model,
    messages=[{"role": "user", "content": "FDC가 뭐야?"}]
)
"""

print(CLAUDE_FINETUNE_EXAMPLE)